<a href="https://colab.research.google.com/github/gav-ip/ML-zero/blob/main/transformer_addition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import jax
from jax import random
import jax.numpy as jnp
import numpy as np
import flax.linen as nn
import random as py_random

In [21]:
USE_GPU = False
device = jax.devices("cpu")[0] if not USE_GPU else jax.devices()[0]
n_examples = 100000
block_size = 4
batch_size = 4
eval_iters = 200
n_embed = 32
n_heads = 4
n_blocks = 4

print(device)

cpu:0


In [22]:
vocab = {0:'0', 1:'1', 2:'2', 3:'3', 4:'4', 5:'5', 6:'6', 7:'7', 8:'8', 9:'9', 10:'+', 11:'=', 12:' '}
len(vocab)

13

In [23]:
stoi = {vocab[i]:i for i, ch in enumerate(vocab)}
itos = {i:vocab[i] for i, ch in enumerate(vocab)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[int(i)] for i in l])

print(encode("8+2 =10 "))
print(decode(encode("8+7 =15 ")))

[8, 10, 2, 12, 11, 1, 0, 12]
8+7 =15 


In [24]:
def build_key(current_key, min_val, max_val):
  # Split the key to generate 'a' and 'b', and get a new key for the next iteration
  subkey_a, subkey_b, next_key = random.split(current_key, 3)
  # Generate a single integer value using .item()
  a = random.randint(subkey_a, shape=(), minval=min_val, maxval=max_val).item()
  b = random.randint(subkey_b, shape=(), minval=min_val, maxval=max_val).item()
  return a, b, next_key

In [25]:
def cateogrize_sum(current_key, split, min_sum_val, max_sum_val):
  target_digit_count = int(n_examples * split)

  collected_examples = []

  while (len(collected_examples) < target_digit_count):
    # Call build_key to get a single (a, b) pair and an updated key
    a, b, current_key = build_key(current_key, min_sum_val, max_sum_val) # Always generate a and b between 0-999

    s = a + b # s is now a scalar integer

    # Check if the sum 's' falls into the desired range (one-digit, two-digit, or three-digit)
    # and ensure sum s is not too large (max 3 digits for 0-999)
    if min_sum_val <= s <= max_sum_val:
      c_reversed = str(s)[::-1] # Now 's' is scalar, str(s) works as expected
      key_str = f"{a:>3}+{b:>3}={c_reversed:<4}"
      collected_examples.append(key_str)

  return collected_examples, current_key # Return the list of strings and the updated key

In [26]:
def build_data():
  initial_key = random.PRNGKey(0) # Initialize JAX random key once

  one_digit_sums = []
  two_digit_sums = []
  three_digit_sums = [] # Renamed from 'other_digit_sums' for clarity

  # Pass and update the JAX random key correctly for each categorization call
  # The min_sum_val and max_sum_val here define the range for the *sum's digits*
  one_digit_sums, initial_key = cateogrize_sum(initial_key, 0.25, 0, 9)
  two_digit_sums, initial_key = cateogrize_sum(initial_key, 0.25, 10, 99)
  three_digit_sums, initial_key = cateogrize_sum(initial_key, 0.5, 100, 999)

  # Combine all generated examples
  all_examples = one_digit_sums + two_digit_sums + three_digit_sums

  # Shuffle the combined list to mix the categories randomly
  py_random.shuffle(all_examples)

  text = all_examples # 'text' is now a list of formatted strings

  return text

# Call build_data and unpack the results
text = build_data()

In [27]:
encoded_text = [encode(t) for t in text]
# Convert the list of encoded lists into a JAX 2D array
data = jnp.array(encoded_text, device=device)

print(f"Data shape: {data.shape}")
data

Data shape: (100000, 12)


Array([[12, 12,  2, ..., 12, 12, 12],
       [12, 12,  1, ..., 12, 12, 12],
       [12,  6,  7, ...,  9, 12, 12],
       ...,
       [12,  2,  1, ...,  8, 12, 12],
       [12,  6,  5, ...,  9, 12, 12],
       [12,  2,  1, ...,  5, 12, 12]], dtype=int32)

In [28]:
print(decode(data[1]))

  1+  7=8   


In [29]:
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [30]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = jnp.randint(0, len(data) - block_size, (batch_size,))
    x = jnp.stack([data[i:i+block_size] for i in ix])
    y = jnp.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)
